# 🏨 Explorador de Datos Hoteleros

### Herramienta interactiva para análisis y exploración de datos

Esta aplicación permite a cualquier usuario:

- 📂 Cargar un archivo CSV  
- ⚙️ Aplicar preprocesamiento básico  
- 📊 Realizar análisis exploratorio (EDA)  
- 🎯 Analizar el riesgo de cancelaciones  

---

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, HTML
import io

sns.set_style("whitegrid")

In [9]:
df = None

In [10]:
display(HTML("""
<style>
h1, h2, h3 {
    color: #1f4e79;
}
.widget-label {
    font-weight: bold;
}
.output_area {
    padding: 10px;
}
</style>
"""))

## 📂 Carga de datos

Sube un archivo CSV para comenzar el análisis.

In [ ]:
# ============================================================
# 📂 CARGA ROBUSTA (UPLOAD + RUTA) — COMPATIBLE CON VOILÀ
# ============================================================

import io

# -------- Widgets --------
uploader = widgets.FileUpload(accept='.csv', multiple=False)
btn_upload = widgets.Button(description="📂 Cargar archivo", button_style='success')

ruta = widgets.Text(description="Ruta CSV:", placeholder="C:\\ruta\\archivo.csv")
btn_ruta = widgets.Button(description="Cargar desde ruta", button_style='info')

btn_reset = widgets.Button(description="Eliminar dataset", button_style='danger')

output_upload = widgets.Output()

# 🎯 Target / problema
target_selector = widgets.Dropdown(options=[], description='Target:')
problem_selector = widgets.RadioButtons(
    options=[('Clasificación','classification'),('Regresión','regression')],
    description='Problema:'
)
target_output = widgets.Output()

# 🔧 Toggle debug
debug_toggle = widgets.Checkbox(value=False, description="Mostrar debug")

# -------- Helpers --------
def _read_csv_bytes(content_bytes):
    """Lee CSV con varios intentos (coma, ;, latin1)"""
    try:
        return pd.read_csv(io.BytesIO(content_bytes))
    except:
        try:
            return pd.read_csv(io.BytesIO(content_bytes), sep=';')
        except:
            return pd.read_csv(io.BytesIO(content_bytes), encoding='latin1')

def _update_all_dropdowns(cols):
    """Actualiza dropdowns de todo el notebook (si existen)"""
    # Target
    target_selector.options = cols
    if cols:
        target_selector.value = cols[-1]
    # Preprocess
    try:
        col_selector.options = cols
    except:
        pass
    # EDA
    try:
        var_x.options = cols
        var_y.options = cols
        if len(cols) > 1:
            var_x.value, var_y.value = cols[0], cols[1]
        elif len(cols) == 1:
            var_x.value = var_y.value = cols[0]
    except:
        pass
    # Insights
    try:
        var_analisis.options = cols
    except:
        pass
    # Calls seguras
    try:
        actualizar_eda()
    except:
        pass
    try:
        actualizar_modelo()
    except:
        pass

def _preview_df():
    """Muestra preview sin truncar columnas"""
    try:
        with pd.option_context('display.max_columns', None):
            display(df.head())
    except:
        display(df.head())

# -------- Cargar desde Upload --------
def cargar_desde_upload(b):
    global df
    with output_upload:
        output_upload.clear_output()
        
        if debug_toggle.value:
            print("🔍 uploader.value:", type(uploader.value), uploader.value)
        
        if not uploader.value:
            print("⚠️ No se ha cargado archivo (Upload vacío)")
            print("👉 Sube el archivo, espera 1–2s y luego haz click en 'Cargar archivo'")
            return
        
        try:
            # Soporta dict (algunas versiones) y tuple (otras)
            if isinstance(uploader.value, dict):
                archivo = list(uploader.value.values())[0]
            elif isinstance(uploader.value, (list, tuple)):
                archivo = uploader.value[0]
            else:
                print("❌ Formato desconocido de uploader.value")
                return
            
            content = archivo.get('content', None)
            if content is None:
                print("❌ No se encontró 'content' en el archivo subido")
                return
            
            df = _read_csv_bytes(content)
            
            print("✅ Dataset cargado (Upload)")
            print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
            
            cols = df.columns.tolist()
            _update_all_dropdowns(cols)
            _preview_df()
        
        except Exception as e:
            print("❌ Error al cargar desde upload:")
            print(e)

# -------- Cargar desde Ruta (fallback seguro) --------
def cargar_desde_ruta(b):
    global df
    with output_upload:
        output_upload.clear_output()
        
        path = ruta.value.strip()
        if not path:
            print("⚠️ Ingresa una ruta válida")
            return
        
        try:
            try:
                df = pd.read_csv(path)
            except:
                try:
                    df = pd.read_csv(path, sep=';')
                except:
                    df = pd.read_csv(path, encoding='latin1')
            
            print("✅ Dataset cargado (Ruta)")
            print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
            
            cols = df.columns.tolist()
            _update_all_dropdowns(cols)
            _preview_df()
        
        except Exception as e:
            print("❌ Error al cargar desde ruta:")
            print(e)

# -------- Target --------
def seleccionar_target(change):
    global target, problem_type
    with target_output:
        target_output.clear_output()
        
        target = target_selector.value
        problem_type = problem_selector.value
        
        if target is None:
            print("⚠️ Selecciona variable target")
            return
        
        print(f"🎯 Target: {target}")
        print(f"🧠 Problema: {problem_type}")
        
        # hint básico
        try:
            n_unique = df[target].nunique()
            if problem_type == 'classification':
                print(f"Clases detectadas: {n_unique}")
        except:
            pass

# -------- Reset --------
def resetear(b):
    global df, target
    with output_upload:
        output_upload.clear_output()
        
        df = None
        target = None
        
        target_selector.options = []
        try:
            col_selector.options = []
        except:
            pass
        try:
            var_x.options = []
            var_y.options = []
        except:
            pass
        try:
            var_analisis.options = []
        except:
            pass
        
        print("🗑️ Dataset eliminado. Puedes cargar uno nuevo.")

# -------- Eventos --------
btn_upload.on_click(cargar_desde_upload)
btn_ruta.on_click(cargar_desde_ruta)
btn_reset.on_click(resetear)

target_selector.observe(seleccionar_target, names='value')
problem_selector.observe(seleccionar_target, names='value')

# -------- UI --------
upload_box = widgets.VBox([
    widgets.HTML("<b>📂 Carga de datos</b>"),
    
    widgets.HTML("<b>Subir archivo (Upload)</b>"),
    uploader,
    btn_upload,
    
    widgets.HTML("<hr><b>o cargar desde ruta (recomendado si Upload falla)</b>"),
    ruta,
    btn_ruta,
    
    widgets.HTML("<hr>"),
    debug_toggle,
    output_upload,
    
    widgets.HTML("<hr><b>🎯 Selección de variable objetivo</b>"),
    target_selector,
    
    widgets.HTML("<b>🧠 Tipo de problema</b>"),
    problem_selector,
    target_output,
    
    widgets.HTML("<hr>"),
    btn_reset
])

## ⚙️ Preprocesamiento de datos

En esta sección puedes limpiar tus datos de manera flexible.

Tienes 3 modos:

- 🤖 Automático: limpieza rápida con parámetros estándar  
- 🔧 Global: limpieza configurable para todo el dataset  
- 🎯 Por columna: limpieza específica por variable  

Además, puedes trabajar con:

- Valores nulos  
- Outliers  
- Valores imposibles

## 🤖 Modo automático

Este modo aplica una limpieza estándar:

- Elimina columnas con más de 50% de nulos  
- Rellena valores numéricos con la mediana  
- Rellena variables categóricas con "Unknown"  
- Convierte fechas automáticamente  

No elimina outliers ni valores extremos.

Ideal para exploración rápida.

### ℹ️ ¿Qué es el IQR?

El IQR (Rango Intercuartílico) detecta valores extremos.

- Identifica el rango donde están la mayoría de datos  
- Detecta valores muy alejados  

👉 IQR bajo → más agresivo  
👉 IQR alto → más flexible  

Valor típico: 1.5

### ℹ️ Valores imposibles

Son datos que no tienen sentido real.

Ejemplos:
- Valores negativos donde no deberían existir  
- Temperaturas irreales  
- Porcentajes mayores a 100  

Puedes definir rangos válidos

In [12]:
# ============================================================
# ⚙️ PREPROCESAMIENTO + TRAZABILIDAD
# ============================================================

# 🔐 requiere: df, target
# 🧾 historial global
try:
    historial
except NameError:
    historial = []

# widgets
modo = widgets.RadioButtons(
    options=[('Global','global'),('Por columna','col')],
    description='Modo:'
)

nulos_check = widgets.Checkbox(value=True, description='Aplicar nulos')
strategy = widgets.Dropdown(
    options=[('Mediana','median'),('Promedio','mean'),('Eliminar filas','drop')],
    description='Estrategia'
)

outliers_check = widgets.Checkbox(value=False, description='Aplicar outliers')
iqr = widgets.FloatSlider(value=1.5, min=0.5, max=3, step=0.1, description='IQR')

imposibles_check = widgets.Checkbox(value=False, description='Aplicar imposibles')
min_val = widgets.FloatText(value=0, description='Mín')
max_val = widgets.FloatText(value=100, description='Máx')

col_selector = widgets.Dropdown(options=[], description='Columna')

btn_global = widgets.Button(description="Aplicar global", button_style='primary')
btn_col = widgets.Button(description="Aplicar columna", button_style='info')

btn_hist = widgets.Button(description="Ver historial", button_style='warning')
btn_clear_hist = widgets.Button(description="Limpiar historial", button_style='danger')

output = widgets.Output()

# helpers
def log(accion, columna, params):
    historial.append({
        "timestamp": pd.Timestamp.now(),
        "accion": accion,
        "columna": columna,
        "parametros": params
    })

def mostrar_historial():
    if len(historial) == 0:
        print("📭 Sin acciones registradas")
    else:
        display(pd.DataFrame(historial))

def actualizar_cols():
    if df is not None:
        col_selector.options = df.columns.tolist()

# ============================================================
# GLOBAL
# ============================================================

def aplicar_global(b):
    global df
    with output:
        output.clear_output()
        
        if df is None:
            print("⚠️ Carga datos primero")
            return
        
        df = df.copy()
        
        # NULOS
        if nulos_check.value:
            for col in df.columns:
                if col != target and es_numerica(df[col]):
                    if strategy.value == 'median':
                        df[col].fillna(df[col].median(), inplace=True)
                    elif strategy.value == 'mean':
                        df[col].fillna(df[col].mean(), inplace=True)
                    elif strategy.value == 'drop':
                        df.dropna(inplace=True)
                    
                    log("nulos", col, {"estrategia": strategy.value})
        
        # OUTLIERS
        if outliers_check.value:
            for col in df.select_dtypes(include=np.number):
                if col != target:
                    q1,q3 = df[col].quantile([0.25,0.75])
                    i = q3 - q1
                    df = df[(df[col] >= q1 - iqr.value*i) & (df[col] <= q3 + iqr.value*i)]
                    log("outliers", col, {"iqr": iqr.value})
        
        # IMPOSIBLES
        if imposibles_check.value:
            for col in df.select_dtypes(include=np.number):
                if col != target:
                    df = df[(df[col] >= min_val.value) & (df[col] <= max_val.value)]
                    log("imposibles", col, {"min": min_val.value, "max": max_val.value})
        
        print("✅ Global aplicado")
        actualizar_cols()
        display(df.head())

# ============================================================
# POR COLUMNA
# ============================================================

def aplicar_columna(b):
    global df
    with output:
        output.clear_output()
        
        col = col_selector.value
        
        if col == target:
            print("⚠️ No modificar target")
            return
        
        # NULOS
        if nulos_check.value:
            if strategy.value == 'median':
                df[col].fillna(df[col].median(), inplace=True)
            elif strategy.value == 'mean':
                df[col].fillna(df[col].mean(), inplace=True)
            elif strategy.value == 'drop':
                df.dropna(subset=[col], inplace=True)
            
            log("nulos", col, {"estrategia": strategy.value})
        
        # OUTLIERS
        if outliers_check.value:
            q1,q3 = df[col].quantile([0.25,0.75])
            i = q3 - q1
            df = df[(df[col] >= q1 - iqr.value*i) & (df[col] <= q3 + iqr.value*i)]
            log("outliers", col, {"iqr": iqr.value})
        
        # IMPOSIBLES
        if imposibles_check.value:
            df = df[(df[col] >= min_val.value) & (df[col] <= max_val.value)]
            log("imposibles", col, {"min": min_val.value, "max": max_val.value})
        
        print(f"✅ Columna procesada: {col}")
        actualizar_cols()
        display(df.head())

# ============================================================
# HISTORIAL
# ============================================================

def ver_hist(b):
    with output:
        output.clear_output()
        mostrar_historial()

def limpiar_hist(b):
    global historial
    historial = []
    print("🧹 Historial limpiado")

# eventos
btn_global.on_click(aplicar_global)
btn_col.on_click(aplicar_columna)
btn_hist.on_click(ver_hist)
btn_clear_hist.on_click(limpiar_hist)

# UI
config = widgets.VBox([
    widgets.HTML("<b>🔵 Nulos</b>"), nulos_check, strategy,
    widgets.HTML("<hr><b>🟠 Outliers</b>"), outliers_check, iqr,
    widgets.HTML("<hr><b>🔴 Imposibles</b>"), imposibles_check, min_val, max_val
])

pre_box = widgets.VBox([
    modo,
    widgets.HTML("<hr>"),
    config,
    col_selector,
    widgets.HBox([btn_global, btn_col]),
    widgets.HTML("<hr><b>📜 Historial</b>"),
    widgets.HBox([btn_hist, btn_clear_hist]),
    output
])

actualizar_cols()

## 📊 Análisis Exploratorio de Datos (EDA)

Explora estadísticas y comprende la estructura del dataset.

In [13]:
# ============================================================
# 📊 EDA FINAL (SEGMENTACIÓN MULTIVARIABLE PROFESIONAL)
# ============================================================

output_eda = widgets.Output()

# ============================================================
# 🎯 VARIABLES PRINCIPALES
# ============================================================

var_x = widgets.Dropdown(description='Variable:')
var_y = widgets.Dropdown(description='Target:')

tipo_grafico = widgets.RadioButtons(
    options=[
        "📊 Histograma (numérica)",
        "📦 Boxplot (numérica vs target)",
        "📈 Scatter (numérica vs numérica)",
        "📊 Barplot (categórica vs target)"
    ],
    description='Gráfico:'
)

# ============================================================
# 🧩 SELECCIÓN DE VARIABLES DE SEGMENTACIÓN
# ============================================================

seg_vars_selector = widgets.SelectMultiple(
    options=[],
    description='Segmentar por:'
)

btn_confirmar_seg = widgets.Button(
    description="Confirmar variables",
    button_style='info'
)

segmentacion_container = widgets.VBox([])

# ============================================================
# 🎛️ CONFIGURACIÓN DINÁMICA
# ============================================================

segment_config = {}  # guarda widgets por variable

def es_numerica(col):
    return df[col].dtype in ['int64','float64']

def generar_controles_segmentacion(vars_seleccionadas):
    
    global segment_config
    
    controles = []
    segment_config = {}
    
    for col in vars_seleccionadas:
        
        if es_numerica(col):
            
            min_w = widgets.FloatText(description='Min')
            max_w = widgets.FloatText(description='Max')
            btn = widgets.Button(description=f"Aplicar {col}", button_style='success')
            
            segment_config[col] = {
                'tipo': 'num',
                'min': min_w,
                'max': max_w,
                'activo': False
            }
            
            def crear_callback(c):
                def aplicar(b):
                    segment_config[c]['activo'] = True
                return aplicar
            
            btn.on_click(crear_callback(col))
            
            controles.append(widgets.VBox([
                widgets.HTML(f"<b>🔢 {col}</b>"),
                min_w,
                max_w,
                btn
            ]))
        
        else:
            
            selector = widgets.SelectMultiple(
                options=df[col].dropna().unique().tolist(),
                description='Clases'
            )
            
            btn = widgets.Button(description=f"Aplicar {col}", button_style='success')
            
            segment_config[col] = {
                'tipo': 'cat',
                'selector': selector,
                'activo': False
            }
            
            def crear_callback(c):
                def aplicar(b):
                    segment_config[c]['activo'] = True
                return aplicar
            
            btn.on_click(crear_callback(col))
            
            controles.append(widgets.VBox([
                widgets.HTML(f"<b>🏷️ {col}</b>"),
                selector,
                btn
            ]))
    
    segmentacion_container.children = controles

# ============================================================
# 🔄 CONFIRMAR VARIABLES
# ============================================================

def confirmar_segmentacion(b):
    if df is None:
        return
    
    generar_controles_segmentacion(seg_vars_selector.value)

btn_confirmar_seg.on_click(confirmar_segmentacion)

# ============================================================
# 📊 FUNCIÓN PRINCIPAL
# ============================================================

def analizar(b):
    with output_eda:
        output_eda.clear_output()
        
        if df is None:
            print("⚠️ Carga datos primero")
            return
        
        dff = df.copy()
        
        # ====================================================
        # 🔥 APLICAR TODAS LAS SEGMENTACIONES
        # ====================================================
        
        for col, config in segment_config.items():
            
            if not config['activo']:
                continue
            
            if config['tipo'] == 'num':
                
                min_val = config['min'].value
                max_val = config['max'].value
                
                dff = dff[
                    (dff[col] >= min_val) &
                    (dff[col] <= max_val)
                ]
            
            else:
                
                valores = config['selector'].value
                
                if len(valores) > 0:
                    dff = dff[dff[col].isin(valores)]
        
        # ====================================================
        # 📊 GRÁFICOS
        # ====================================================
        
        col = var_x.value
        target = var_y.value
        
        plt.figure(figsize=(6,4))
        
        tipo = tipo_grafico.value
        
        if "Histograma" in tipo:
            if not es_numerica(col):
                print("⚠️ Solo numérica")
                return
            sns.histplot(dff[col], kde=True)
        
        elif "Boxplot" in tipo:
            if not es_numerica(col):
                print("⚠️ Variable numérica requerida")
                return
            sns.boxplot(x=dff[target], y=dff[col])
        
        elif "Scatter" in tipo:
            if not es_numerica(col) or not es_numerica(target):
                print("⚠️ Ambas deben ser numéricas")
                return
            sns.scatterplot(x=dff[col], y=dff[target])
        
        elif "Barplot" in tipo:
            sns.barplot(x=dff[col], y=dff[target])
            plt.xticks(rotation=45)
        
        plt.title(f"{tipo} - {col}")
        plt.tight_layout()
        plt.show()
        
        # ====================================================
        # 💡 INSIGHTS
        # ====================================================
        
        print("\n💡 Insights:")
        print(f"- Filas después de segmentación: {len(dff)}")
        
        if es_numerica(target):
            print(f"- Promedio de {target}: {dff[target].mean():.2f}")

# ============================================================
# EVENTO
# ============================================================

boton = widgets.Button(description="Generar análisis", button_style='primary')
boton.on_click(analizar)

# ============================================================
# 🔄 ACTUALIZAR UI
# ============================================================

def actualizar_eda():
    if df is None:
        return
    
    cols = df.columns.tolist()
    
    opciones = []
    for c in cols:
        if es_numerica(c):
            opciones.append((f"🔢 {c}", c))
        else:
            opciones.append((f"🏷️ {c}", c))
    
    var_x.options = opciones
    var_y.options = opciones
    seg_vars_selector.options = opciones

# ============================================================
# UI FINAL
# ============================================================

eda_box = widgets.VBox([
    
    widgets.HTML("<h3>📊 Exploración de datos</h3>"),
    
    var_x,
    var_y,
    
    widgets.HTML("<hr><b>🎨 Tipo de gráfico</b>"),
    tipo_grafico,
    
    widgets.HTML("<hr><b>🧩 Selección de segmentación</b>"),
    seg_vars_selector,
    btn_confirmar_seg,
    
    widgets.HTML("<hr><b>⚙️ Configuración de segmentación</b>"),
    segmentacion_container,
    
    widgets.HTML("<hr>"),
    boton,
    output_eda
])

# 🔥 IMPORTANTE
actualizar_eda()

In [14]:
# ============================================================
# 🧩 TABS FINALES (SIN INSIGHTS)
# ============================================================

tabs = widgets.Tab()

# 🔥 SOLO 3 TABS
tabs.children = [
    upload_box,   # 📂 Carga
    pre_box,      # ⚙️ Preprocesamiento
    eda_box       # 📊 EDA
]

# Nombres de tabs
tabs.set_title(0, "📂 Carga")
tabs.set_title(1, "⚙️ Preprocess")
tabs.set_title(2, "📊 EDA")

display(tabs)